# SDialog Tutorial: Serving Agents

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/agents/3.serving.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

Once an agent works, you can let people (or other programs) talk to it. SDialog serves any agent through an **OpenAI/Ollama-compatible REST API**, so chat apps like Open WebUI and any OpenAI client can use it as if it were a model, and it can also chat with real users **on WhatsApp**.

What you will learn:
- Serve an agent with tools, and talk to it with the OpenAI Python client
- How conversations, parallel users and rate limits work
- Connect a chat app (Open WebUI)
- Put your agent on WhatsApp

## 1. Setup

In [1]:
# Setup the environment depending on whether we are running in Google Colab or Jupyter Notebook
from IPython import get_ipython


if "google.colab" in str(get_ipython()):
    print("Running on CoLab")

    # Installing Ollama (if you are not planning to use Ollama, you can just comment these lines to speed up the installation)
    !apt-get install zstd
    !curl -fsSL https://ollama.com/install.sh | sh

    # Installing sdialog
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .
    %cd ..
else:
    print("Running in Jupyter Notebook")
    # Little hack to avoid the "OSError: Background processes not supported." error in Jupyter notebooks"
    import os
    get_ipython().system = os.system

Running in Jupyter Notebook


If you plan to use an Ollama backend locally, start the server in the background. If you use OpenAI/Bedrock/HF instead, you can skip this cell.

In [2]:
# Let's run the ollama server in the background
!OLLAMA_KEEP_ALIVE=-1 ollama serve > /dev/null 2>&1 &
!sleep 5

Set the global model used by SDialog (any "BACKEND:MODEL" string works, e.g. `"openai:gpt-4.1"`). We use a Qwen3 model, which supports tool calling:

In [3]:
import sdialog

sdialog.config.llm("ollama:qwen3:14b")
# sdialog.config.llm("openai:gpt-4.1")  # e.g. OpenAI GPT-4.1 (set the OPENAI_API_KEY environment variable)

## 2. An agent to serve

Let's build a small support agent for an online shop, with a tool to look up orders (see the [tools tutorial](1.agents_with_tools_and_thoughts.ipynb) for more):

In [4]:
from sdialog.agents import Agent
from sdialog.personas import SupportAgent

ORDERS = {"A1001": "shipped, arriving tomorrow", "A1002": "being prepared", "A1003": "delivered on Monday"}


def get_order_status(order_id: str) -> dict:
    """Get the status of an order by its order id (e.g. "A1001")."""
    return {"order_id": order_id, "status": ORDERS.get(order_id.upper(), "not found")}


support = Agent(SupportAgent(name="Ava", product_scope="orders of an online shop", communication_style="brief and friendly"),
                tools=[get_order_status],
                name="Support")

support("Hi! Where is my order A1001?")

'Your order A1001 is shipped and arriving tomorrow. Let me know if you need further assistance!'

## 3. Serve it

`agent.serve(port=...)` starts an OpenAI/Ollama-compatible server. In a script, it runs until you stop it. In a notebook, we run it in the background with `Server.serve_in_thread()`, so we can keep using the notebook to talk to it:

In [5]:
import time
from sdialog.server import Server

PORT = 1333

Server.serve_in_thread(support, port=PORT,
                       # no rate limits for this local test (by default, 30 requests per minute per client)
                       max_requests_per_minute=None, max_requests_per_hour=None)
time.sleep(5)  # give the server a few seconds to start

The server lists the agents it serves as models, named after them:

In [6]:
from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")  # any key works: the server does not check it

model = client.models.list().data[0].id
model

'Support:latest'

## 4. Talk to it like any OpenAI model

Now any OpenAI client can chat with our agent, tools included:

In [7]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Hi! Can you check order A1002 for me?"}],
)
print(response.choices[0].message.content)

Your order A1002 is currently being prepared. It should be shipped soon!


By default, the server is **stateless**, like OpenAI's API: each request carries the whole conversation, so many users can share the same agent. To continue a conversation, send its history:

In [8]:
messages = [{"role": "user", "content": "Hi! Can you check order A1002 for me?"},
            {"role": "assistant", "content": response.choices[0].message.content},
            {"role": "user", "content": "And what about A1003?"}]
response = client.chat.completions.create(model=model, messages=messages)
print(response.choices[0].message.content)

Your order A1003 was delivered on Monday. Let me know if you need anything else!


### Many users at once

The server answers up to `max_workers` requests at the same time (8 by default), each on a copy of the agent that shares its model, so users don't wait for each other. This also means the agent's **tools may run at the same time** for different users, so they must be thread-safe (pass `max_workers=1` to process one request at a time).

Let's send four conversations at once:

In [9]:
from concurrent.futures import ThreadPoolExecutor


def ask(order_id):
    answer = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": f"What's the status of order {order_id}?"}])
    return order_id, answer.choices[0].message.content


start = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    for order_id, answer in pool.map(ask, ["A1001", "A1002", "A1003", "A9999"]):
        print(f"{order_id}: {answer}\n")
print(f"4 conversations answered in {time.time() - start:.1f}s")

A1001: Your order A1001 is shipped and arriving tomorrow. Let me know if you need further assistance!

A1002: Your order A1002 is currently being prepared. Thank you for your patience!

A1003: Your order A1003 has been delivered on Monday. Let me know if you need anything else!

A9999: I couldn't find any information on order A9999. Could you please check the order number and try again?

4 conversations answered in 3.2s


> 💡 **Rate limits.** To keep anyone who can reach the server from flooding it (or running up your LLM costs), each client can send up to 30 chat requests per minute and 300 per hour by default (`max_requests_per_minute` and `max_requests_per_hour`). Extra requests get an HTTP 429 error, which clients like the OpenAI SDK retry automatically. We disabled them above since all our requests come from this notebook.

## 5. Chat apps

Since the server speaks the Ollama and OpenAI APIs, chat apps can use it too. For instance, to chat with the agent in [Open WebUI](https://github.com/open-webui/open-webui), serve it (e.g. `support.serve(port=1333)` in a script) and launch Open WebUI pointing to it:

```bash
docker run -d -e OLLAMA_BASE_URL=http://host.docker.internal:1333 \
              -p 3030:8080 -v open-webui:/app/backend/data --name open-webui \
              ghcr.io/open-webui/open-webui:main
```

Then open http://localhost:3030 and pick the `Support` model. To serve several agents on the same port (selected by name), use `Server.serve([agent1, agent2], port=1333)`.

## 6. WhatsApp

Agents can also chat with real users on WhatsApp, through Meta's WhatsApp Cloud API, with `agent.serve_whatsapp()`. Each user (phone number) gets their own conversation, saved between messages. It needs a few things a notebook can't easily provide: credentials from a Meta developer app, a public HTTPS address for Meta to send messages to (e.g. an [ngrok](https://ngrok.com/) tunnel), and a server that keeps running. So it is best started from a script, such as [`whatsapp_bot.py`](whatsapp_bot.py), which serves this tutorial's support agent:

```bash
pip install sdialog pyngrok
export WHATSAPP_ACCESS_TOKEN=...      # from your Meta app (WhatsApp > API Setup)
export WHATSAPP_PHONE_NUMBER_ID=...
export WHATSAPP_APP_SECRET=...        # App settings > Basic
export WHATSAPP_VERIFY_TOKEN=any-random-string
python whatsapp_bot.py
```

The [Serving Agents on WhatsApp](https://sdialog.readthedocs.io/en/latest/sdialog/index.html#serving-agents-on-whatsapp) documentation explains, step by step, how to get these with a free test number, and the options for sessions, rate limits and reacting to events (e.g. alerting a person when a safety orchestrator fires).

## Learn more

- [Serving Agents via REST API](https://sdialog.readthedocs.io/en/latest/sdialog/index.html#serving-agents-via-rest-api): all the server options.
- [Testing remote systems with simulated users](https://github.com/idiap/sdialog#-testing-remote-systems-with-simulated-users): the other way around, using SDialog agents to test a deployed OpenAI-compatible system.